In [ ]:
%%capture
import os, time, random, warnings
import numpy as np
import pandas as pd
from datetime import datetime, timedelta, timezone
from dataclasses import dataclass, field

warnings.filterwarnings("ignore")

# !pip install mosek
# !pip install polygon-api-client

lic_path = "/content/mosek.lic"
os.environ["MOSEKLM_LICENSE_FILE"] = lic_path
import mosek
import mosek.fusion as mf
from polygon import RESTClient

In [ ]:
RUSSELL_2000_SECTOR_WEIGHTS = {
    "Industrials":              0.1863,
    "Health Care":              0.1723,
    "Financials":               0.1682,
    "Information Technology":   0.1610,
    "Consumer":                 0.0995,
    "Energy":                   0.0575,
    "Real Estate":              0.0523,
    "Materials":                0.0477,
    "Utilities":                0.0286,
    "Communication Services":   0.0266,
}
# total = 0
# for i in RUSSELL_2000_SECTOR_WEIGHTS.values():
#     total += i

# print(total)
HOLDINGS_INFO = {
    # ETFs / Indices
    'PSCI': {'name': 'Invesco S&P SmallCap Industrials ETF', 'expense_ratio': 0.0029, 'asset_type': 'index', 'sector': 'Industrials'},
    'PSCT': {'name': 'Invesco S&P SmallCap Information Technology ETF','expense_ratio': 0.0029, 'asset_type': 'index', 'sector': 'Information Technology'},
    'PSCM': {'name': 'Invesco S&P SmallCap Materials ETF', 'expense_ratio': 0.0029, 'asset_type': 'index', 'sector': 'Materials'},
    'PSCU': {'name': 'Invesco S&P SmallCap Utilities ETF', 'expense_ratio': 0.0029, 'asset_type': 'index', 'sector': 'Utilities'},
    'RSPG': {'name': 'Invesco S&P 500 Equal Weight Energy ETF', 'expense_ratio': 0.004, 'asset_type': 'index', 'sector': 'Energy'},
    'RSPF': {'name': 'Invesco S&P 500 Equal Weight Financials ETF', 'expense_ratio': 0.004, 'asset_type': 'index', 'sector': 'Financials'},
    'PSR' : {'name': 'Invesco Active U.S. Real Estate ETF', 'expense_ratio': 0.0055,'asset_type': 'index', 'sector': 'Real Estate'},
    'PSCC': {'name': 'Invesco S&P SmallCap Consumer Discretionary ETF', 'expense_ratio': 0.0029, 'asset_type': 'index', 'sector': 'Consumer'},
    'PSCH': {'name': 'Invesco S&P SmallCap Health Care ETF', 'expense_ratio': 0.0029, 'asset_type': 'index', 'sector': 'Health Care'},

    # Stocks
    'CALY': {'name': 'Callaway Golf / Topgolf Callaway Brands', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Consumer'},
    'UAA' : {'name': 'Under Armour', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Consumer'},
    'MSGE': {'name': 'Madison Square Garden Entertainment','conviction': 8,'asset_type': 'stock', 'sector': 'Consumer'},
    'CELH': {'name': 'Celsius Holdings', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Consumer'},
    'TPB': {'name': 'Turning Point Brands', 'conviction': 8, 'asset_type': 'stock', 'sector': 'Consumer'},

    'STEP': {'name': 'StepStone Group', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Financials'},
    'NMFC': {'name': 'New Mountain Finance Corp', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Financials'},
    'AXS':  {'name': 'Axis Capital', 'conviction': 7, 'asset_type':'stock', 'sector':'Financials'},
    'WULF': {'name': 'Terawulf Inc', 'conviction': 7, 'asset_type':'stock', 'sector':'Financials'},

    'ENSG': {'name': 'The Ensign Group','conviction': 5, 'asset_type': 'stock', 'sector': 'Health Care'},
    'EHC' : {'name': 'Encompass Health', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Health Care'},
    'PRVA': {'name': 'Privia Health', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Health Care'},
    'NKTR': {'name': 'Nektar Therapeutics','conviction': 9,'asset_type': 'stock', 'sector': 'Health Care'},
    'ALHC': {'name': 'Allignment Healthcare', 'conviction': 9, 'asset_type': 'stock', 'sector': 'Health Care'},
    'PSNL': {'name': 'Personalis Inc', 'conviction': 7, 'asset_type': 'stock', 'sector': 'Health Care'},

    'NXT' : {'name': 'Nextracker', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Information Technology'},
    'LASR': {'name': 'nLIGHT','conviction': 5, 'asset_type': 'stock', 'sector': 'Information Technology'},
    'ESE': {'name': 'Esco Technologies Inc', 'conviction': 7, 'asset_type': 'stock', 'sector': 'Information Technology'},
    'SIMO': {'name': 'Silicon Motion Technology', 'conviction': 8, 'asset_type': 'stock', 'sector': 'Information Technology'},

    'ACHR': {'name': 'Archer Aviation','conviction': 5, 'asset_type': 'stock', 'sector': 'Industrials'},
    'FLY' : {'name': 'Firefly', 'conviction': 8, 'asset_type': 'stock', 'sector': 'Industrials'},
    'CDNL' : {'name': 'Cardinal Infrastructure', 'conviction': 8, 'asset_type': 'stock', 'sector': 'Industrials'},
    'LUNR' : {'name': 'Intuitive Machines', 'conviction': 7, 'asset_type': 'stock', 'sector': 'Industrials'},
    'FMC' : {'name': 'FMC', 'conviction': 7, 'asset_type': 'stock', 'sector': 'Industrials'},

    'EFXT' : {'name': 'Enerflex', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Energy'},
    'LBRT' : {'name': 'Liberty Energy', 'conviction': 6, 'asset_type': 'stock', 'sector': 'Energy'},

    'SGML' : {'name': 'Sigma Lithium', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Materials'},

    'IRDM' : {'name': 'Iridium', 'conviction': 5, 'asset_type': 'stock', 'sector': 'Communication Services'}
}

In [ ]:
@dataclass
class Config:
    MAX_STOCK_POSITION: float = 0.04
    MAX_INDEX_POSITION: float = 0.08
    MIN_STOCK_POSITION: float = 0.005
    # For cash: factor in $0 for ACUR, $14226.77 for ELLO, $12050.73
    CASH_RESERVE: float = 0.015
    ETF_SLEEVE_MIN: float = 0.25
    ETF_SLEEVE_MAX: float = 0.40
    SECTOR_HARD_BOUND: float = 0.02

    CONVICTION_LAMBDA: float = 10.0
    N_FACTORS: int = 8
    LOOKBACK_DAYS: int = 400
    RETURN_FREQUENCY: str = "W"
    MIN_HISTORY_WEEKS: int = 12
    BENCHMARK_TICKER: str = "IWM"
    POLYGON_LIMIT: int = 50000
    INTER_TICKER_SLEEP: float = 0.1
    PRICE_CSV_PATH: str = "daily_prices_bufc-2.csv"
    TRACKING_ERROR_DAYS: int = 126  # ~6 months trading days

CFG = Config()

### Data functions

In [ ]:
# %% Polygon Data Pipeline
class PolygonThrottle:
    """Proactive token-bucket rate limiter for Polygon free tier (5 calls/min)."""
    def __init__(self, calls_per_minute: int = 4):
        self.interval = 60.0 / calls_per_minute
        self.call_times: list = []
        self.calls_per_minute = calls_per_minute

    def wait(self):
        now = time.time()
        self.call_times = [t for t in self.call_times if now - t < 60.0]
        if len(self.call_times) >= self.calls_per_minute:
            oldest = self.call_times[0]
            sleep_for = 60.0 - (now - oldest) + 0.5
            if sleep_for > 0:
                print(f"  Throttle: waiting {sleep_for:.1f}s "
                      f"({len(self.call_times)}/{self.calls_per_minute} calls in window)")
                time.sleep(sleep_for)
        self.call_times.append(time.time())

_throttle = PolygonThrottle(calls_per_minute=4)

def get_aggs_with_backoff(client, **kwargs):
    max_retries = 5
    base = 15.0
    for i in range(max_retries):
        _throttle.wait()
        try:
            return client.get_aggs(**kwargs)
        except Exception as e:
            msg = str(e).lower()
            if (("429" in msg) or ("rate" in msg) or ("retry" in msg)
                    or ("too many" in msg)) and i < max_retries - 1:
                delay = min(base * (2 ** i), 120.0)
                print(f"  429 despite throttle; backing off {delay:.0f}s (attempt {i+1}/{max_retries})")
                time.sleep(delay)
                continue
            raise

def sleep_with_jitter(base: float, jitter: float = 0.4) -> None:
    time.sleep(base + random.random() * jitter)


def fetch_polygon_adj_close_series(client, ticker, start_date, end_date, polygon_limit=50000):
    """
    Fetch daily adjusted closes for a ticker over an allowed range only.
    """
    aggs = get_aggs_with_backoff(
        client,
        ticker=ticker,
        multiplier=1,
        timespan="day",
        from_=start_date,
        to=end_date,
        adjusted=True,
        sort="asc",
        limit=polygon_limit,
    )

    if not aggs:
        return pd.Series(name=ticker, dtype=float)

    tmp = pd.DataFrame(
        [
            {
                "date": datetime.fromtimestamp(a.timestamp / 1000, tz=timezone.utc),
                ticker: a.close,
            }
            for a in aggs
        ]
    )

    tmp["date"] = pd.to_datetime(tmp["date"]).dt.tz_localize(None).dt.normalize()
    tmp = tmp.drop_duplicates(subset=["date"], keep="last")
    tmp = tmp.set_index("date").sort_index()

    return tmp[ticker]


def _last_accessible_trading_day() -> pd.Timestamp:
    """
    Conservative last day we should request from Polygon on this plan.
    Always use the most recent *completed prior* trading day, never today.
    """
    now_et = pd.Timestamp.now(tz="America/New_York")
    day = now_et.normalize().tz_localize(None) - pd.Timedelta(days=1)

    while day.weekday() >= 5:  # Saturday/Sunday
        day -= pd.Timedelta(days=1)

    return pd.Timestamp(day).normalize()

def _load_csv(path):
    if not os.path.exists(path):
        return pd.DataFrame()

    df = pd.read_csv(path, index_col=0, parse_dates=True)
    df.index = pd.to_datetime(df.index).normalize()
    df.index.name = "date"
    df = df[~df.index.duplicated(keep="last")].sort_index()
    return df


def _save_csv(df, path):
    df = df.copy()
    df.index = pd.to_datetime(df.index).normalize()
    df = df[~df.index.duplicated(keep="last")].sort_index()
    df.index.name = "date"
    df.to_csv(path)
    print(f"  Saved {len(df)} rows × {len(df.columns)} tickers → {path}")


def download_returns(client, holdings_info, config):
    """
    CSV-cached price loader with incremental append, restricted to the last
    config.LOOKBACK_DAYS calendar days only.

    Returns
    -------
    returns : pd.DataFrame
    prices  : pd.DataFrame
    ppy     : int
    """
    csv_path = getattr(config, "PRICE_CSV_PATH", "daily_prices_bufc.csv")
    tickers = list(holdings_info.keys()) + [config.BENCHMARK_TICKER]

    # ---- hard cap history window ----
    history_days = min(int(config.LOOKBACK_DAYS), 400)

    end_dt = datetime.today()
    start_dt = end_dt - timedelta(days=history_days)

    start_str = start_dt.strftime("%Y-%m-%d")
    end_str = end_dt.strftime("%Y-%m-%d")
    lookback_start = pd.Timestamp(start_str).normalize()
    last_trade = _last_accessible_trading_day()

    print(f"  Using capped history window: {start_str} → {end_str} ({history_days} calendar days)")

    prices = _load_csv(csv_path)

    # trim old CSV immediately so you never carry stale history forward
    if not prices.empty:
        prices = prices[prices.index >= lookback_start].sort_index()

    needs_full_fetch = prices.empty
    csv_modified = False

    if not needs_full_fetch:
        initial_shape = prices.shape
        csv_last_date = prices.index.max()

        print(f"  CSV loaded: {initial_shape[0]} rows × {initial_shape[1]} cols, last date = {csv_last_date.date()}")
        print(f"  Last US trading day (approx) = {last_trade.date()}")

        # ------------------------------------------------------------
        # 1) Fetch any missing tickers only inside the capped window
        # ------------------------------------------------------------
        missing_tickers = [t for t in tickers if t not in prices.columns]
        if missing_tickers:
            print(f"  Fetching {len(missing_tickers)} new tickers: {missing_tickers}")
            for tk in missing_tickers:
                try:
                    s = fetch_polygon_adj_close_series(
                        client, tk, start_str, end_str, config.POLYGON_LIMIT
                    )
                    if not s.empty:
                        s.index = pd.to_datetime(s.index).normalize()
                        s = s[s.index >= lookback_start]
                        # OLD: prices = prices.combine_first(s.to_frame())
                        combined = pd.concat([prices, s.to_frame()])
                        prices = combined.groupby(combined.index).last().sort_index()
                        csv_modified = True
                    else:
                        print(f"    WARNING: no data for {tk}")
                except Exception as e:
                    print(f"    WARNING: failed to fetch {tk}: {e}")
                sleep_with_jitter(config.INTER_TICKER_SLEEP)

        prices = prices[~prices.index.duplicated(keep="last")].sort_index()
        prices = prices[prices.index >= lookback_start]

        if not prices.empty:
            csv_last_date = prices.index.max()

        # ------------------------------------------------------------
        # 2) Append only the missing recent dates
        # ------------------------------------------------------------
        if (not prices.empty) and (csv_last_date < last_trade):
            append_start_dt = max(csv_last_date + pd.Timedelta(days=1), lookback_start)
            append_start = append_start_dt.strftime("%Y-%m-%d")

            print(f"  Appending {append_start} → {end_str} for {len(tickers)} tickers ...")

            append_series = []
            for tk in tickers:
                try:
                    s = fetch_polygon_adj_close_series(
                        client, tk, append_start, end_str, config.POLYGON_LIMIT
                    )
                    if not s.empty:
                        s.index = pd.to_datetime(s.index).normalize()
                        s = s[s.index >= lookback_start]
                        append_series.append(s.rename(tk))
                except Exception as e:
                    print(f"    WARNING: failed append fetch for {tk}: {e}")
                sleep_with_jitter(config.INTER_TICKER_SLEEP)

            if append_series:
                append_block = pd.concat(append_series, axis=1)
                append_block.index = pd.to_datetime(append_block.index).normalize()
                append_block = append_block[~append_block.index.duplicated(keep="last")].sort_index()
                append_block = append_block[append_block.index >= lookback_start]

                # OLD (broken): update() won't touch rows not in append_block;
                # combine_first() won't overwrite existing (NaN) rows in prices
                # prices.update(append_block)
                # prices = prices.combine_first(append_block)

                # NEW: concat with append_block LAST so its values win on duplicate dates,
                # then group-by index keeping the last (freshest) non-NaN value per cell
                combined = pd.concat([prices, append_block])
                combined = combined.groupby(combined.index).last()  # last() prefers non-NaN over NaN
                prices = combined[~combined.index.duplicated(keep="last")].sort_index()
                prices = prices[prices.index >= lookback_start]
                csv_modified = True

            print(f"  After append: {len(prices)} rows, last = {prices.index.max().date()}")
        else:
            print("  CSV is current — no API calls needed.")

        if csv_modified:
            _save_csv(prices, csv_path)
            print("  ✓ CSV updated")
        else:
            print("  ✓ CSV unchanged")

    else:
        # ------------------------------------------------------------
        # No CSV yet: full fetch, but only for the capped window
        # ------------------------------------------------------------
        print(f"  No CSV at '{csv_path}' — full download from Polygon.")
        print(f"  Fetching {len(tickers)} tickers [{start_str} → {end_str}] ...")

        series_list = []
        for idx, tk in enumerate(tickers, 1):
            print(f"    [{idx}/{len(tickers)}] {tk} ...", end=" ", flush=True)
            try:
                s = fetch_polygon_adj_close_series(
                    client, tk, start_str, end_str, config.POLYGON_LIMIT
                )
                if s.empty:
                    print("no data")
                else:
                    s.index = pd.to_datetime(s.index).normalize()
                    s = s[s.index >= lookback_start]
                    series_list.append(s.rename(tk))
                    print(f"{len(s)} days")
            except Exception as e:
                print(f"failed ({e})")
            sleep_with_jitter(config.INTER_TICKER_SLEEP)

        if not series_list:
            raise ValueError("No price data from Polygon for any ticker.")

        prices = pd.concat(series_list, axis=1).sort_index()
        prices = prices[prices.index >= lookback_start]
        _save_csv(prices, csv_path)
        print(f"  ✓ CSV created: {len(prices)} rows × {len(prices.columns)} cols")

    # ----------------------------------------------------------------
    # Final enforce: keep only last 400 calendar days of prices
    # ----------------------------------------------------------------
    prices = prices[prices.index >= lookback_start].sort_index()

    # resample if needed
    resample_map = {"W": "W-FRI", "M": "ME", "D": None}
    rule = resample_map.get(config.RETURN_FREQUENCY)
    if rule:
        prices = prices.resample(rule).last()

    # log returns from capped window only
    returns = np.log(prices / prices.shift(1)).dropna(how="all")

    # do not drop shorter-history tickers aggressively; treat them like IPO names
    min_obs = getattr(config, "MIN_HISTORY_WEEKS", 12)
    counts = returns.count()
    valid = counts[counts >= min_obs].index.tolist()

    dropped = [c for c in returns.columns if c not in valid]
    if dropped:
        print(f"  WARNING: dropping tickers with < {min_obs} return obs: {dropped}")

    returns = returns[valid]
    prices = prices[[c for c in prices.columns if c in valid]]

    # backfill missing return history with zeros so younger tickers / shorter
    # history names behave like IPO names
    n_missing = returns.isna().sum().sum()
    if n_missing > 0:
        pct_missing = 100 * n_missing / returns.size
        print(f"  Filling {n_missing} NaN returns ({pct_missing:.1f}%) with 0.0")
        returns = returns.fillna(0.0)

    returns = returns.dropna(how="all")

    ppy = {"D": 252, "W": 52, "M": 12}[config.RETURN_FREQUENCY]
    print(f"  {len(returns)} {config.RETURN_FREQUENCY}-periods, {len(returns.columns)} tickers, annualization={ppy}")

    return returns, prices, ppy

In [ ]:
# # %% Polygon Data Pipeline
# class PolygonThrottle:
#     def __init__(self, calls_per_minute=4):
#         self.call_times = []
#         self.max_calls = calls_per_minute

#     def wait(self):
#         now = time.time()
#         self.call_times = [t for t in self.call_times if now - t < 60]
#         if len(self.call_times) >= self.max_calls:
#             sleep_for = 60 - (now - self.call_times[0]) + 0.5
#             if sleep_for > 0:
#                 print(f"  Throttle: {sleep_for:.0f}s")
#                 time.sleep(sleep_for)
#         self.call_times.append(time.time())

# _throttle = PolygonThrottle()


# def _fetch_one(client, ticker, start, end, limit=50000):
#     """Fetch daily adj close for one ticker. Returns dict {date_str: close}."""
#     for attempt in range(4):
#         _throttle.wait()
#         try:
#             aggs = client.get_aggs(
#                 ticker=ticker, multiplier=1, timespan="day",
#                 from_=start, to=end, adjusted=True, sort="asc", limit=limit,
#             )
#             if not aggs:
#                 return {}
#             return {
#                 pd.Timestamp(a.timestamp, unit="ms").normalize(): a.close
#                 for a in aggs
#             }
#         except Exception as e:
#             if "429" in str(e).lower() and attempt < 3:
#                 wait = 15 * (2 ** attempt)
#                 print(f"    429 on {ticker}, wait {wait}s")
#                 time.sleep(wait)
#             else:
#                 print(f"    Error on {ticker}: {e}")
#                 return {}
#     return {}


# def _load_csv(path):
#     if not os.path.exists(path):
#         return pd.DataFrame()
#     df = pd.read_csv(path, index_col=0, parse_dates=True)
#     df.index = pd.to_datetime(df.index).normalize()
#     df.index.name = "date"
#     return df


# def _save_csv(df, path):
#     df = df.sort_index()
#     df.index.name = "date"
#     df.to_csv(path)


# def download_returns(client, holdings_info, config):
#     """Fetch prices with CSV cache, return (log_returns, periods_per_year)."""
#     csv_path = config.PRICE_CSV_PATH
#     tickers = list(holdings_info.keys()) + [config.BENCHMARK_TICKER]

#     end_dt = datetime.today()
#     start_dt = end_dt - timedelta(days=int(config.LOOKBACK_DAYS * 1.6))
#     start_str, end_str = start_dt.strftime("%Y-%m-%d"), end_dt.strftime("%Y-%m-%d")

#     prices = _load_csv(csv_path)
#     old_shape = prices.shape if not prices.empty else (0, 0)

#     if prices.empty:
#         # ── Full fetch ──
#         print(f"  No CSV — full download ({len(tickers)} tickers)...")
#         all_data = {}
#         for i, tk in enumerate(tickers, 1):
#             print(f"    [{i}/{len(tickers)}] {tk}", end=" ", flush=True)
#             data = _fetch_one(client, tk, start_str, end_str, config.POLYGON_LIMIT)
#             if data:
#                 all_data[tk] = data
#                 print(f"({len(data)} days)")
#             else:
#                 print("(no data)")
#             time.sleep(config.INTER_TICKER_SLEEP)
#         if not all_data:
#             raise ValueError("No data from Polygon.")
#         prices = pd.DataFrame(all_data).sort_index()
#         prices.index = pd.to_datetime(prices.index).normalize()
#         _save_csv(prices, csv_path)
#         print(f"  ✓ CSV created: {prices.shape[0]} rows × {prices.shape[1]} cols")
#     else:
#         csv_last = prices.index.max()
#         print(f"  CSV: {old_shape[0]} rows × {old_shape[1]} cols, last={csv_last.date()}")

#         # Figure out what needs fetching
#         missing_tickers = [t for t in tickers if t not in prices.columns]
#         stale_tickers = [t for t in tickers if t in prices.columns]
#         need_append = csv_last.date() < end_dt.date()

#         modified = False

#         # ── New tickers: full range ──
#         if missing_tickers:
#             print(f"  Fetching {len(missing_tickers)} new tickers...")
#             for tk in missing_tickers:
#                 data = _fetch_one(client, tk, start_str, end_str, config.POLYGON_LIMIT)
#                 if data:
#                     s = pd.Series(data, name=tk)
#                     s.index = pd.to_datetime(s.index).normalize()
#                     prices = prices.join(s, how="outer")
#                     modified = True
#                     print(f"    {tk}: {len(data)} days")
#                 else:
#                     print(f"    {tk}: no data")
#                 time.sleep(config.INTER_TICKER_SLEEP)

#         # ── Stale tickers: append from last date + 1 ──
#         if need_append and stale_tickers:
#             append_from = (csv_last + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
#             print(f"  Updating {len(stale_tickers)} tickers from {append_from}...")
#             new_data = {}
#             for tk in stale_tickers:
#                 data = _fetch_one(client, tk, append_from, end_str, config.POLYGON_LIMIT)
#                 if data:
#                     new_data[tk] = data
#                 time.sleep(config.INTER_TICKER_SLEEP)
#             if new_data:
#                 patch = pd.DataFrame(new_data)
#                 patch.index = pd.to_datetime(patch.index).normalize()
#                 # Combine: patch overwrites any overlap, adds new rows
#                 prices = prices.combine_first(patch)
#                 prices.update(patch)  # ensure patch values win on overlap
#                 modified = True
#                 new_dates = set(patch.index) - set(prices.index.intersection(pd.to_datetime(list(set(prices.index)))))
#                 print(f"    Patched {len(patch)} date-rows across {len(new_data)} tickers")

#         prices = prices.sort_index()

#         if modified:
#             _save_csv(prices, csv_path)
#             print(f"  ✓ CSV updated: {old_shape[0]}→{len(prices)} rows, "
#                   f"{old_shape[1]}→{len(prices.columns)} cols, "
#                   f"last={prices.index.max().date()}")
#         else:
#             print(f"  ✓ CSV already current (last={csv_last.date()})")

#     # ── Trim & resample ──
#     prices = prices[prices.index >= pd.Timestamp(start_str)]
#     rule = {"W": "W-FRI", "M": "ME"}.get(config.RETURN_FREQUENCY)
#     if rule:
#         prices = prices.resample(rule).last()

#     returns = np.log(prices / prices.shift(1)).dropna(how="all")

#     min_obs = getattr(config, "MIN_HISTORY_WEEKS", 12)
#     valid = returns.columns[returns.count() >= min_obs]
#     dropped = set(returns.columns) - set(valid)
#     if dropped:
#         print(f"  Dropped (<{min_obs} obs): {dropped}")
#     returns = returns[valid].dropna(how="all")

#     n_nan = returns.isna().sum().sum()
#     if n_nan > 0:
#         print(f"  Filling {n_nan} NaN ({n_nan/returns.size*100:.1f}%) with 0")
#         returns = returns.fillna(0.0)

#     ppy = {"D": 252, "W": 52, "M": 12}[config.RETURN_FREQUENCY]
#     print(f"  {len(returns)} {config.RETURN_FREQUENCY}-periods, {len(returns.columns)} tickers")
#     return returns, ppy

### Return + Covariance

In [ ]:
def build_factor_covariance(returns, n_factors=8, ppy=52):
    """PCA factor model → (Sigma_annual, tickers)."""
    tickers = returns.columns.tolist()
    R = returns.values.copy()
    R = np.nan_to_num(R, nan=0.0)

    mu = R.mean(axis=0)
    sigma = R.std(axis=0, ddof=1)
    sigma[sigma < 1e-10] = 1e-10
    Z = (R - mu) / sigma

    U, S, Vt = np.linalg.svd(Z, full_matrices=False)
    K = min(n_factors, len(S))
    B_std = Vt[:K].T
    factor_var_std = (S[:K] ** 2) / (len(R) - 1)

    B = B_std * sigma[:, None]
    explained = B @ np.diag(factor_var_std) @ B.T
    total_var = np.diag(np.cov(R.T))
    residual_var = np.maximum(total_var - np.diag(explained), 1e-10)

    factor_var_annual = factor_var_std * ppy
    residual_var_annual = residual_var * ppy

    Sigma = B @ np.diag(factor_var_annual) @ B.T + np.diag(residual_var_annual)
    return Sigma, tickers


# %% Conviction Targets
def build_conviction_targets(holdings, config):
    stocks = {t: h for t, h in holdings.items() if h["asset_type"] == "stock"}
    total_conv = sum(h["conviction"] for h in stocks.values())
    stock_sleeve = 1.0 - config.CASH_RESERVE - (config.ETF_SLEEVE_MIN + config.ETF_SLEEVE_MAX) / 2
    return {t: (h["conviction"] / total_conv) * stock_sleeve for t, h in stocks.items()}


### Diagnose feasibility, Post-Hoc analysis

In [ ]:
# %% Feasibility Diagnostics
def diagnose_feasibility(tickers, holdings, benchmark_sector_weights, config):
    sectors = sorted(benchmark_sector_weights.keys())
    stocks = [t for t in tickers if holdings[t]["asset_type"] == "stock"]
    etfs = [t for t in tickers if holdings[t]["asset_type"] == "index"]

    print("\n=== FEASIBILITY DIAGNOSTICS ===")
    print(f"Stocks: {len(stocks)}, ETFs: {len(etfs)}, Budget: {1 - config.CASH_RESERVE:.2%}")
    print(f"Stock capacity: [{len(stocks)*config.MIN_STOCK_POSITION:.2%}, {len(stocks)*config.MAX_STOCK_POSITION:.2%}]")
    print(f"ETF capacity: [0, {len(etfs)*config.MAX_INDEX_POSITION:.2%}]")
    print(f"ETF sleeve required: [{config.ETF_SLEEVE_MIN:.2%}, {config.ETF_SLEEVE_MAX:.2%}]")

    for sec in sectors:
        rw = benchmark_sector_weights.get(sec, 0)
        sec_stocks = [t for t in stocks if holdings[t]["sector"] == sec]
        sec_etfs = [t for t in etfs if holdings[t]["sector"] == sec]
        max_cap = len(sec_stocks) * config.MAX_STOCK_POSITION + len(sec_etfs) * config.MAX_INDEX_POSITION
        needed = rw - config.SECTOR_HARD_BOUND
        if max_cap < needed:
            print(f"  ✗ {sec}: max capacity {max_cap:.2%} < needed {needed:.2%}")


def compute_tracking_error(client, holdings_info, weights, config):
    """
    Historical fixed-weight tracking error and information ratio
    versus the benchmark.

    Notes
    -----
    - Uses current optimized weights as a constant-weight historical overlay.
      This is descriptive, not a true out-of-sample backtest.
    - Missing history for newer tickers is treated as 0 return before inception.
    """
    csv_path = getattr(config, "PRICE_CSV_PATH", "daily_prices_bufc.csv")
    prices = _load_csv(csv_path)

    if prices.empty:
        print("  ✗ No cached prices for TE calculation")
        return {}

    prices.index = pd.to_datetime(prices.index).normalize()
    benchmark = config.BENCHMARK_TICKER

    if benchmark not in prices.columns:
        print(f"  ✗ Benchmark {benchmark} not in price data")
        return {}

    holding_tickers = [t for t in weights if t != benchmark and t in prices.columns]
    missing = [t for t in weights if t != benchmark and t not in prices.columns]
    if missing:
        print(f"  TE warning: missing from price data: {missing}")

    if not holding_tickers:
        print("  ✗ No portfolio tickers found in price data")
        return {}

    cols = holding_tickers + [benchmark]
    px = prices[cols].copy().sort_index()

    # Forward-fill occasional missing prices, then compute returns.
    # Before inception, returns remain NaN and are then treated as 0.0.
    px = px.ffill()
    daily_rets = px.pct_change()

    # Treat pre-inception / short-history missing returns like IPO names: 0 return
    daily_rets = daily_rets.fillna(0.0)

    # Fixed-weight portfolio return series
    port_rets = pd.Series(0.0, index=daily_rets.index)
    for t in holding_tickers:
        port_rets += float(weights.get(t, 0.0)) * daily_rets[t]

    bench_rets = daily_rets[benchmark]
    active_rets = port_rets - bench_rets

    windows = {"3-month": 63, "6-month": 126}
    results = {}

    print()
    for label, days in windows.items():
        if len(active_rets) < days:
            print(f"  {label}: only {len(active_rets)} days available, need {days} — skipped")
            continue

        pr = port_rets.iloc[-days:]
        br = bench_rets.iloc[-days:]
        ar = active_rets.iloc[-days:]

        # Annualized tracking error
        te = ar.std(ddof=1) * np.sqrt(252)

        # Correct IR numerator: annualized average active return
        active_ann = ar.mean() * 252

        # Cumulative returns over the window
        port_cum = (1.0 + pr).prod() - 1.0
        bench_cum = (1.0 + br).prod() - 1.0

        # More coherent relative excess return over the whole window
        rel_excess = ((1.0 + pr).prod() / (1.0 + br).prod()) - 1.0

        ir = rel_excess / te if te > 1e-12 else np.nan

        results[label] = {
            "tracking_error": te,
            "info_ratio": ir,
            "annualized_active_return": active_ann,
            "portfolio_return": port_cum,
            "benchmark_return": bench_cum,
            "relative_excess_return": rel_excess,
            "active_rets": ar,
            "portfolio_rets": pr,
            "benchmark_rets": br,
        }

        print(f"  {label} ({days}d):")
        print(f"    Portfolio return:        {port_cum:+.2%}")
        print(f"    Benchmark return:       {bench_cum:+.2%}")
        print(f"    Relative excess return: {rel_excess:+.2%}")
        print(f"    Ann. active return:     {active_ann:+.2%}")
        print(f"    Tracking error:         {te:.2%}")
        print(f"    Information ratio:      {ir:.2f}")

    return results


# %% Display
def display_results(weights, holdings, benchmark_sector_weights, Sigma, tickers, conv_targets, active_returns=None, te=None):
    if weights is None:
        print("No feasible solution.")
        return

    w_arr = np.array([weights.get(t, 0) for t in tickers])

    rows = []
    for t in sorted(weights, key=lambda x: -weights[x]):
        w = weights[t]
        if w < 1e-6:
            continue
        h = holdings[t]
        rows.append({"Ticker": t, "Name": h["name"][:30], "Weight": w,
                      "Type": "ETF" if h["asset_type"] == "index" else "STK",
                      "Sector": h["sector"], "Conv": h.get("conviction", "—")})
    df = pd.DataFrame(rows)
    print("\n" + "=" * 80)
    print("OPTIMAL PORTFOLIO WEIGHTS")
    print("=" * 80)
    df["Weight"] = df["Weight"].map(lambda x: f"{x:.2%}")
    print(df.to_string(index=False))

    print("\n" + "=" * 80)
    print("SECTOR BREAKDOWN")
    print("=" * 80)
    print(f"{'Sector':<25} {'Portfolio':>10} {'Russell':>10} {'Active':>10}")
    print("-" * 55)
    for sec in sorted(benchmark_sector_weights.keys()):
        rw = benchmark_sector_weights[sec]
        pw = sum(weights.get(t, 0) for t in weights if holdings[t]["sector"] == sec)
        print(f"{sec:<25} {pw:>10.2%} {rw:>10.2%} {pw - rw:>+10.2%}")

    port_var = w_arr @ Sigma @ w_arr
    port_vol = np.sqrt(port_var)
    etf_pct = sum(weights.get(t, 0) for t in weights if holdings[t]["asset_type"] == "index")
    conv_fidelity = np.sqrt(sum((weights.get(t, 0) - conv_targets.get(t, 0)) ** 2 for t in weights))

    print("\n" + "=" * 80)
    print("DIAGNOSTICS")
    print("=" * 80)
    print(f"  Annualized portfolio vol:   {port_vol:.2%}")
    print(f"  ETF sleeve:                 {etf_pct:.2%}")
    print(f"  Conviction fidelity (RMSE): {conv_fidelity:.4f}")


### Portfolio Optimizer

In [ ]:
# %% MOSEK Optimizer
def optimize_portfolio(Sigma, tickers, holdings, benchmark_sector_weights, config):
    import mosek.fusion as mf

    N = len(tickers)
    ticker_idx = {t: i for i, t in enumerate(tickers)}
    stocks = [t for t in tickers if holdings[t]["asset_type"] == "stock"]
    etfs = [t for t in tickers if holdings[t]["asset_type"] == "index"]
    stock_idx = [ticker_idx[t] for t in stocks]
    etf_idx = [ticker_idx[t] for t in etfs]

    conv_targets = build_conviction_targets(holdings, config)
    w_target = np.zeros(N)
    for t, wt in conv_targets.items():
        if t in ticker_idx:
            w_target[ticker_idx[t]] = wt

    try:
        L = np.linalg.cholesky(Sigma)
    except np.linalg.LinAlgError:
        eigvals = np.linalg.eigvalsh(Sigma)
        Sigma += np.eye(N) * (abs(eigvals.min()) + 1e-8)
        L = np.linalg.cholesky(Sigma)

    sectors = sorted(benchmark_sector_weights.keys())
    sector_matrix = np.zeros((len(sectors), N))
    for j, sec in enumerate(sectors):
        for t in tickers:
            if holdings[t]["sector"] == sec:
                sector_matrix[j, ticker_idx[t]] = 1.0

    budget = 1.0 - config.CASH_RESERVE

    with mf.Model("portfolio") as M:
        w = M.variable("w", N, mf.Domain.greaterThan(0.0))
        M.constraint("budget", mf.Expr.sum(w), mf.Domain.equalsTo(budget))

        for i in stock_idx:
            M.constraint(f"cap_stk_{i}", w.index(i), mf.Domain.lessThan(config.MAX_STOCK_POSITION))
        for i in etf_idx:
            M.constraint(f"cap_etf_{i}", w.index(i), mf.Domain.lessThan(config.MAX_INDEX_POSITION))
        if config.MIN_STOCK_POSITION > 0:
            for i in stock_idx:
                M.constraint(f"floor_{i}", w.index(i), mf.Domain.greaterThan(config.MIN_STOCK_POSITION))

        etf_sum = mf.Expr.add([w.index(i) for i in etf_idx])
        M.constraint("etf_lo", etf_sum, mf.Domain.greaterThan(config.ETF_SLEEVE_MIN))
        M.constraint("etf_hi", etf_sum, mf.Domain.lessThan(config.ETF_SLEEVE_MAX))

        for j, sec in enumerate(sectors):
            sec_w = mf.Expr.dot(sector_matrix[j].tolist(), w)
            rw = benchmark_sector_weights.get(sec, 0.0)
            M.constraint(f"sec_lo_{sec}", sec_w, mf.Domain.greaterThan(rw - config.SECTOR_HARD_BOUND))
            M.constraint(f"sec_hi_{sec}", sec_w, mf.Domain.lessThan(rw + config.SECTOR_HARD_BOUND))

        t_var = M.variable("t_var", 1, mf.Domain.greaterThan(0.0))
        half = M.variable("half", 1, mf.Domain.equalsTo(0.5))
        Lw = mf.Expr.mul(mf.Matrix.dense(L.T), w)
        M.constraint("var_cone", mf.Expr.vstack(t_var, half, Lw), mf.Domain.inRotatedQCone())

        t_conv = M.variable("t_conv", 1, mf.Domain.greaterThan(0.0))
        half2 = M.variable("half2", 1, mf.Domain.equalsTo(0.5))
        diff = mf.Expr.sub(w, w_target.tolist())
        M.constraint("conv_cone", mf.Expr.vstack(t_conv, half2, diff), mf.Domain.inRotatedQCone())

        obj = mf.Expr.add(t_var, mf.Expr.mul(config.CONVICTION_LAMBDA, t_conv))
        M.objective("obj", mf.ObjectiveSense.Minimize, obj)
        M.solve()

        if M.getProblemStatus() != mf.ProblemStatus.PrimalAndDualFeasible:
            print(f"⚠ Problem status: {M.getProblemStatus()}")
            diagnose_feasibility(tickers, holdings, benchmark_sector_weights, config)
            return None

        weights = np.array(w.level())
        return {tickers[i]: weights[i] for i in range(N)}


### Pipeline Function

In [ ]:
def run_optimizer(client, config=None, holdings_info=None, benchmark_sector_weights=None):
    if holdings_info is None:
        holdings_info = HOLDINGS_INFO
    if config is None:
        config = Config()
    if benchmark_sector_weights is None:
        benchmark_sector_weights = RUSSELL_2000_SECTOR_WEIGHTS

    print("1/5 Downloading prices...")
    returns, prices, ppy = download_returns(client, holdings_info, config)

    # only keep names that actually survived the capped return build
    available = [t for t in holdings_info if t in returns.columns]
    if len(available) < 3:
        raise ValueError(f"Only {len(available)} tickers have usable data — need ≥ 3.")

    holdings_used = {t: holdings_info[t] for t in available}
    prices = prices[[c for c in prices.columns if c in available + [config.BENCHMARK_TICKER] if c in prices.columns]]

    print(f"     {len(returns)} periods × {len(available)} tickers with data")

    print("2/5 Building PCA factor covariance...")
    holding_rets = returns[available].copy()

    # this now uses only the capped window and the zero-filled shorter histories
    Sigma, cov_tickers = build_factor_covariance(holding_rets, config.N_FACTORS, ppy)
    print(f"     {config.N_FACTORS} factors, condition number: {np.linalg.cond(Sigma):.0f}")

    print("3/5 Building conviction targets...")
    conv_targets = build_conviction_targets(holdings_used, config)

    print("4/5 Solving with MOSEK...")
    weights = optimize_portfolio(
        Sigma,
        cov_tickers,
        holdings_used,
        benchmark_sector_weights,
        config,
    )

    print("5/5 Computing 6-month tracking error...")
    te, active = None, None
    if weights:
        te, active = compute_tracking_error(client, holdings_used, weights, config)

    display_results(
        weights,
        holdings_used,
        benchmark_sector_weights,
        Sigma,
        cov_tickers,
        conv_targets,
        active,
        te,
    )

    return weights, Sigma, cov_tickers, active, prices

### Run Optimizer

In [ ]:
client = RESTClient(api_key="6liGmp5MZGmi_q41LN0rx5arI5e8UZEt")

weights, Sigma, tickers, active, prices = run_optimizer(client)

1/5 Downloading prices...
  Using capped history window: 2025-03-18 → 2026-04-22 (400 calendar days)
  CSV loaded: 275 rows × 38 cols, last date = 2026-04-21
  Last US trading day (approx) = 2026-04-21
  CSV is current — no API calls needed.
  ✓ CSV unchanged
  Filling 101 NaN returns (4.7%) with 0.0
  57 W-periods, 38 tickers, annualization=52
     57 periods × 37 tickers with data
2/5 Building PCA factor covariance...
     8 factors, condition number: 536
3/5 Building conviction targets...
4/5 Solving with MOSEK...
5/5 Computing 6-month tracking error...

  3-month (63d):
    Portfolio return:        +21.18%
    Benchmark return:       +5.62%
    Relative excess return: +14.73%
    Ann. active return:     +55.69%
    Tracking error:         9.71%
    Information ratio:      1.52
  6-month (126d):
    Portfolio return:        +35.86%
    Benchmark return:       +13.94%
    Relative excess return: +19.23%
    Ann. active return:     +35.83%
    Tracking error:         9.31%
    Informa

In [ ]:
def build_share_trade_sheet(
    weights,
    prices,
    portfolio_value,
    current_shares=None,
    output_path="trade_sheet.csv",
    round_lots=False,
    lot_size=1,
    include_zero_trades=False,
):
    if not weights:
        raise ValueError("weights is empty.")
    if prices is None or prices.empty:
        raise ValueError("prices DataFrame is empty.")
    if portfolio_value <= 0:
        raise ValueError("portfolio_value must be positive.")

    current_shares = current_shares or {}

    latest_row = prices.sort_index().iloc[-1]
    trade_date = prices.sort_index().index[-1]

    rows = []
    for ticker, target_weight in weights.items():
        if ticker not in latest_row.index:
            continue

        px = latest_row[ticker]
        if pd.isna(px) or px <= 0:
            continue

        curr_sh = float(current_shares.get(ticker, 0.0))

        # Infer target shares from target weight and latest price
        target_sh = (target_weight * portfolio_value) / px

        if round_lots:
            target_sh = round(target_sh / lot_size) * lot_size

        trade_sh = target_sh - curr_sh

        # nice signed action label
        if trade_sh > 0:
            action = "BUY"
        elif trade_sh < 0:
            action = "SELL"
        else:
            action = "HOLD"

        rows.append(
            {
                "date": pd.Timestamp(trade_date).date(),
                "ticker": ticker,
                "price": float(px),
                "current_shares": float(curr_sh),
                "target_shares": float(target_sh),
                "trade_shares": float(trade_sh),
                "action": action,
            }
        )

    trade_sheet = pd.DataFrame(rows)

    if trade_sheet.empty:
        raise ValueError("No valid rows were created for the trade sheet.")

    # Optional cleanup
    if not include_zero_trades:
        trade_sheet = trade_sheet[trade_sheet["trade_shares"] != 0].copy()

    # Rounded display values
    trade_sheet["price"] = trade_sheet["price"].round(4)
    trade_sheet["current_shares"] = trade_sheet["current_shares"].round(4)
    trade_sheet["target_shares"] = trade_sheet["target_shares"].round(4)
    trade_sheet["trade_shares"] = trade_sheet["trade_shares"].round(4)

    # Sort by biggest trade first
    trade_sheet["abs_trade"] = trade_sheet["trade_shares"].abs()
    trade_sheet = trade_sheet.sort_values(
        by=["abs_trade", "ticker"], ascending=[False, True]
    ).drop(columns="abs_trade")

    # Clean column order
    trade_sheet = trade_sheet[
        [
            "date",
            "ticker",
            "action",
            "price",
            "current_shares",
            "target_shares",
            "trade_shares",
        ]
    ].reset_index(drop=True)

    trade_sheet.to_csv(output_path, index=False)
    print(f"Saved trade sheet → {output_path}")

    return trade_sheet

In [ ]:
current_shares = {
    "ACHR": 4396, "ALHC": 2727, 'AXS': 527, "CALY":2105, "CDNL":1192, "CELH":678, "EHC":343,
    "ENSG": 189, "FLY": 1288, "FMC": 2125, "LASR": 470, "LBRT": 1387, "LUNR":1575, "MSGE":674,
    "NKTR": 463, "NXT": 297, "PRVA": 1547, "PSCC": 1037, "PSCH": 944, "PSCI": 238, "PSCM": 1163,
    "PSCT": 579, "PSCU": 670, "PSNL": 6286, "PSR": 896, "RSPF": 591, "RSPG": 416, "SGML":1510,
    "SIMO": 363, "STEP": 739, "TPG": 1041, "UAA": 3829, "WULF": 1978
}

trade_sheet = build_share_trade_sheet(
    weights=weights,
    prices=prices,
    portfolio_value=1467497.46,
    current_shares=current_shares,
    output_path="optimized_trade_sheet.csv",
    round_lots=True,
    lot_size=1,
)


Saved trade sheet → optimized_trade_sheet.csv


In [ ]:
trades = pd.read_csv('optimized_trade_sheet.csv')

trades

,date,ticker,action,price,current_shares,target_shares,trade_shares
0,2026-04-24,NMFC,BUY,8.4400,0.0,4932.0,4932.0
1,2026-04-24,EFXT,BUY,23.4800,0.0,1486.0,1486.0
2,2026-04-24,IRDM,BUY,42.9300,0.0,830.0,830.0
3,2026-04-24,PSCM,SELL,104.0339,1163.0,392.0,-771.0
4,2026-04-24,UAA,SELL,6.6100,3829.0,3094.0,-735.0
5,2026-04-24,TPB,BUY,84.5400,0.0,478.0,478.0
6,2026-04-24,PSR,SELL,101.4800,896.0,467.0,-429.0
7,2026-04-24,CALY,SELL,15.2000,2105.0,1735.0,-370.0
8,2026-04-24,SGML,SELL,21.6900,1510.0,1141.0,-369.0
9,2026-04-24,FMC,BUY,17.3500,2125.0,2407.0,282.0
